# 02 · De dónde sale cada cifra de la guía

Respalda las **preguntas** de `guia_exposicion_modelo.pdf`. Cada sección responde una pregunta y vuelve a
calcular su cifra con los datos del repo. Lo que no se puede recalcular aquí (porque necesita Earth Engine o se corrió en Colab)
está al final, con el archivo donde vive.

El modelo Luna contra Marte tiene su propio cuaderno de estudio, más detallado: `ciencia_datos/06_modelo_luna_marte.ipynb`.
El borde levantado está en `01_borde_levantado.ipynb`.

In [1]:
# Preparación: encuentra la raíz del repo (local o Colab) para poder usar el código de model/
import subprocess, sys
from pathlib import Path
REPO, PISTA = "Niettzche/spaceapps-2026", "ciencia_datos/datos/perfiles_radiales.json"
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / PISTA).exists()), None)
if RAIZ is None:      # Colab: el repo es privado, así que clona con el token guardado en el secreto GITHUB_TOKEN
    try:
        from google.colab import userdata
        url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/{REPO}.git"
    except Exception:
        url = f"https://github.com/{REPO}.git"
    subprocess.run(["git", "clone", "--depth", "1", url, "/content/spaceapps-2026"], check=True)
    RAIZ = Path("/content/spaceapps-2026")
sys.path.insert(0, str(RAIZ))

import json, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt, sklearn
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 220)
VALIDACION = (RAIZ / "model" / "VALIDATION.md").read_text(encoding="utf-8")
print(f"repo: {RAIZ.name} · numpy {np.__version__} · pandas {pd.__version__} · scikit-learn {sklearn.__version__}")
from model import forma, validate

repo: spaceapps-2026 · numpy 2.1.3 · pandas 2.2.3 · scikit-learn 1.5.2


## 1. «¿Qué modelo es?» · 4,288 cráteres lunares, 573 marcianos y 78 %

El modelo usa tres medidas de cada cráter (diámetro, profundidad entre diámetro y borde entre diámetro), tomadas con el mismo
detector y a la misma resolución (200 m) en la Luna y en Marte. Estos son los datos con los que se entrena.

In [2]:
ref = pd.read_csv(RAIZ / "ciencia_datos" / "datos" / "hoyas_referencia.csv")
ref = ref[ref.crater.astype(bool) & ~ref.sitio.str.contains("|".join(forma.SITIOS_EXCLUIDOS))].copy()
print(ref.cuerpo.value_counts().to_string(), "\n")
ref.groupby(["cuerpo", "sitio"]).agg(crateres=("diam_km", "size"), diametro_km=("diam_km", "median"), prof_entre_diam=("prof_diam", "median")).round(3)

cuerpo
Luna     4288
Marte     573 



crateres  diametro_km  prof_entre_diam
cuerpo sitio                                                          
Luna   Anillo 80-85°S               3232        1.009            0.061
       Polo sur (85-90°S)           1056        0.957            0.058
Marte  Chryse (Viking 1)              23        1.791            0.027
       Cráter Gale                    90        0.957            0.031
       Cráter Jezero                 104        1.382            0.022
       Gusev (Spirit)                101        1.463            0.029
       Meridiani (Opportunity)       165        1.354            0.025
       Oxia Planum                    90        1.117            0.027

[8 rows x 3 columns]

In [3]:
# La medida que más separa los mundos: profundidad / diámetro (mediana).
# México árido sale con 46 detecciones y 0.033; el informe dice 47 y 0.034. La diferencia es de un cráter que cruza el umbral
# de vegetación según cómo se muestreó el NDVI (PROYECTO.md, paso 3). No cambia la lectura.
mx = pd.read_csv(RAIZ / "ciencia_datos" / "datos" / "hoyas_mexico.csv")
arido = mx[mx.crater.astype(bool) & (mx.ndvi_p90 < 0.35)]
pd.Series({"Luna (polo sur y anillo)": ref[ref.cuerpo == "Luna"].prof_diam.median(), f"México árido ({len(arido)} detecciones)": arido.prof_diam.median(),
           "Marte (6 sitios)": ref[ref.cuerpo == "Marte"].prof_diam.median()}, name="profundidad / diámetro").round(3).to_frame()

,profundidad / diámetro
Luna (polo sur y anillo),0.060
México árido (46 detecciones),0.033
Marte (6 sitios),0.027


**La exactitud.** Se valida por zonas: los cráteres se agrupan en bloques de 50 × 50 km y un bloque entero queda fuera cada
vez, para que el modelo no acierte solo porque vio al cráter vecino. La exactitud es *balanceada* (promedio del acierto en la Luna
y en Marte), porque hay muchos más cráteres lunares. Es la misma función que usa `model/validate.py`; tarda unos segundos.

In [4]:
ref["log_diam"] = np.log10(ref.diam_km); ref["borde_rel"] = ref.borde_m / (ref.diam_km * 1000)
ref["grupo"] = ref.cuerpo + "|" + ref.sitio + "|" + (ref.i // 250).astype(int).astype(str) + "_" + (ref.j // 250).astype(int).astype(str)
filas = []
for nombre, d in [("todos los tamaños", ref), ("solo cráteres de 2 km o más", ref[ref.diam_km >= 2])]:
    for cuales, feats in [("las 3 medidas", forma.VARIABLES), ("solo profundidad / diámetro", ["prof_diam"])]:
        c = validate.cv_grupos(d, feats)
        filas.append({"cráteres": nombre, "medidas": cuales, "Luna": c["n_luna"], "Marte": c["n_marte"],
                      "exactitud balanceada": f"{c['ba']:.1%} ± {c['sd']:.1%}", "acierto Luna": f"{c['luna']:.0%}", "acierto Marte": f"{c['marte']:.0%}"})
exactitud = pd.DataFrame(filas); exactitud

,cráteres,medidas,Luna,Marte,exactitud balanceada,acierto Luna,acierto Marte
0,todos los tamaños,las 3 medidas,4288,573,77.6% ± 0.2%,73%,82%
1,todos los tamaños,solo profundidad / diámetro,4288,573,75.8% ± 0.1%,68%,83%
2,solo cráteres de 2 km o más,las 3 medidas,788,125,77.6% ± 0.7%,72%,83%
3,solo cráteres de 2 km o más,solo profundidad / diámetro,788,125,71.0% ± 0.3%,57%,85%


## 2. «¿Marte sale más plano solo porque su mapa es más borroso?»

La tabla de arriba ya lo responde en parte: con **cráteres de 2 km o más**, donde la resolución de MOLA pesa menos, la exactitud
con las tres medidas se mantiene. Otra comprobación: entrenar sin un sitio de Marte y ver si lo reconoce.

El experimento de emborronar la Luna (0.058 → 0.043) se corrió en Colab; está en `model/NIVEL2.md`, sección B.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
filas = []
for sitio in sorted(ref[ref.cuerpo == "Marte"].sitio.unique()):
    tr, te = ref[ref.sitio != sitio], ref[ref.sitio == sitio]
    m = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=3000)).fit(tr[forma.VARIABLES].to_numpy(float), tr.cuerpo.to_numpy(object))
    filas.append({"sitio de Marte que el modelo no vio": sitio, "cráteres": len(te), "reconocidos como Marte": f"{(m.predict(te[forma.VARIABLES].to_numpy(float)) == 'Marte').mean():.0%}"})
pd.DataFrame(filas)

,sitio de Marte que el modelo no vio,cráteres,reconocidos como Marte
0,Chryse (Viking 1),23,91%
1,Cráter Gale,90,70%
2,Cráter Jezero,104,82%
3,Gusev (Spirit),101,75%
4,Meridiani (Opportunity),165,92%
5,Oxia Planum,90,84%


## 3. «¿Su puntaje pone un análogo real por encima de un hoyo cualquiera?»

La respuesta honesta es **no**. Se comparan cráteres naturales contra valles, presas y dunas (los dos grupos según la etiqueta
de IA con confianza de 0.7 o más). El AUC es la probabilidad de que un cráter al azar puntúe más que un no-cráter al azar:
0.5 es no separar nada.

In [6]:
f = pd.read_csv(RAIZ / "ciencia_datos" / "salidas" / "fichas_integradas.csv")
pf = forma.puntuar(f.diam_km, f.prof_m, f.borde_m)
f["forma_mejor"] = np.maximum(pf.forma_polo_sur_lunar, pf.forma_jezero).values
alta = f.conf_ia >= 0.7
nat, neg = f[alta & f.etiqueta_ia.isin(validate.NATURALES_IA)], f[alta & f.etiqueta_ia.isin(validate.NEGATIVOS_IA)]
print(f"{len(nat)} cráteres naturales contra {len(neg)} valles, presas y dunas")
pd.DataFrame([{"criterio": "puntaje de forma del modelo", "AUC": round(validate.auc(nat.forma_mejor, neg.forma_mejor), 2), "p": round(validate.p_permutacion(nat.forma_mejor, neg.forma_mejor), 2)},
              {"criterio": "profundidad / diámetro sola", "AUC": round(validate.auc(nat.prof_diam, neg.prof_diam), 2), "p": round(validate.p_permutacion(nat.prof_diam, neg.prof_diam), 2)}])

14 cráteres naturales contra 26 valles, presas y dunas


,criterio,AUC,p
0,puntaje de forma del modelo,0.42,0.43
1,profundidad / diámetro sola,0.75,0.01


Con el puntaje de forma el AUC queda por debajo de 0.5 y la `p` es alta: no separa mejor que el azar. Por eso el puntaje no
decide qué es cada sitio; lo deciden la etiqueta contrastada con fuentes y el borde levantado (`01_borde_levantado.ipynb`).

## 4. «¿Qué tan bueno es el detector?» y «¿Escanearon todo el planeta?»

In [7]:
c = validate.cotejo_catalogo()
print(f"Catálogo de cráteres volcánicos de México: {c['n_catalogo']} con fuente, {c['con_coord']} con coordenadas, {c['en_rango']} en el rango del detector")
print(f"  el detector encuentra el {c['detectado']:.0%}; el {c['pasa']:.0%} pasa además el filtro de forma de cráter")
print(f"  no detectados: {c['lago']} con lago, {c['chico']} menores de 0.6 km, {c['otra']} por otras razones\n")
print(f"Candidatos áridos con ficha: {len(f)}, por región:")
print(f.region.value_counts().to_string())

Catálogo de cráteres volcánicos de México: 70 con fuente, 66 con coordenadas, 62 en el rango del detector
  el detector encuentra el 40%; el 32% pasa además el filtro de forma de cráter
  no detectados: 15 con lago, 3 menores de 0.6 km, 22 por otras razones

Candidatos áridos con ficha: 296, por región:
region
Atacama (Chile)              171
México                        47
Arizona (EE. UU.)             45
Arabia occidental (Hiyaz)     33


In [8]:
# Qué son los candidatos áridos según la etiqueta de IA (con cualquier confianza)
vc = f.etiqueta_ia.value_counts()
pd.DataFrame({"candidatos": vc, "%": (100 * vc / len(f)).round(0).astype(int)})

,candidatos,%
etiqueta_ia,,
valle_o_canada,94,32
no_se_puede_decir,57,19
mina_o_cantera,53,18
crater_volcanico,26,9
otro,22,7
hondonada_eolica_o_duna,22,7
presa_o_lago,20,7
crater_impacto,1,0
dolina_karst,1,0


## 5. «¿Cómo usaron la IA?» · 29 de 33

Las etiquetas se contrastaron con fuentes externas donde las hay: minas de OpenStreetMap, el catálogo de volcanes y Meteor Crater.
`ciencia_datos/etiquetado_ia/validacion.txt` tiene el caso por caso.

In [9]:
lineas = [l for l in (RAIZ / "ciencia_datos" / "etiquetado_ia" / "validacion.txt").read_text(encoding="utf-8").splitlines() if " | " in l]
casos = pd.DataFrame([[x.strip() for x in l.split("|")] for l in lineas], columns=["id", "según la fuente", "según la IA", "confianza", "acierto", "fuente"])
print(f"aciertos: {int((casos.acierto == '✓').sum())} de {len(casos)}\n")
print(casos.groupby("según la fuente").acierto.agg(casos="size", aciertos=lambda x: int((x == "✓").sum())).to_string())
print("\nOjo: no hay casos contrastados de valles, presas ni dunas; esas etiquetas no están validadas.")

aciertos: 29 de 33

                  casos  aciertos
según la fuente                  
crater_impacto        1         1
crater_volcanico      6         6
mina_o_cantera       26        22

Ojo: no hay casos contrastados de valles, presas ni dunas; esas etiquetas no están validadas.


## 6. «¿Qué no puede probar la Tierra?»

Los valores del destino están en `data/targets/`, cada uno con su fuente.

In [10]:
filas = []
for d in ("polo_sur_lunar", "jezero"):
    tg = json.loads((RAIZ / "data" / "targets" / f"{d}.json").read_text(encoding="utf-8"))
    for k in ("oscilacion_termica", "pendiente"):
        v = tg["variables"][k]
        filas.append({"destino": tg["nombre"], "variable": k, "valor": v["valor"], "unidad": v["unidad"], "fuente": v["fuente"][:70]})
    filas.append({"destino": tg["nombre"], "variable": "gravedad", "valor": tg["gravedad_ms2"], "unidad": "m/s²", "fuente": "valor del destino"})
print("No se reproducen en la Tierra:", {d: json.loads((RAIZ / 'data' / 'targets' / f'{d}.json').read_text(encoding='utf-8'))['no_reproducible_en_tierra'] for d in ('polo_sur_lunar', 'jezero')})
from model.score import DIMENSIONES
print("Máximo terrestre de oscilación térmica que usa el comparador (valor provisional):", DIMENSIONES["oscilacion_termica"]["rango_tierra"][1], "K")
pd.DataFrame(filas)

No se reproducen en la Tierra: {'polo_sur_lunar': ['gravedad', 'vacio', 'radiacion', 'iluminacion_polar_rasante'], 'jezero': ['gravedad', 'presion_atmosferica', 'radiacion']}
Máximo terrestre de oscilación térmica que usa el comparador (valor provisional): 40.0 K


,destino,variable,valor,unidad,fuente
0,"Región candidata de Artemis, polo sur lunar",oscilacion_termica,176.00,K,"LRO Diviner, mapas polares de temperatura estacional de Williams et al"
1,"Región candidata de Artemis, polo sur lunar",pendiente,8.63,grados,"LRO LOLA polar ldem_80s_80m (NASA), promediado a 200 m"
2,"Región candidata de Artemis, polo sur lunar",gravedad,1.62,m/s²,valor del destino
3,Cráter Jezero (Marte),oscilacion_termica,55.00,K,"Perseverance MEDA, Munguira et al. 2023, JGR Planets 128, e2022JE00755"
4,Cráter Jezero (Marte),pendiente,3.56,grados,"Mosaico HRSC+MOLA Blend DEM 200 m (ESA/NASA, vía USGS Astrogeology)"
5,Cráter Jezero (Marte),gravedad,3.71,m/s²,valor del destino


## 7. Lo que este cuaderno no vuelve a calcular

| Cifra de la guía | Por qué no se recalcula aquí | Dónde está |
|---|---|---|
| 3 de 3 controles negativos y 9 de 21 análogos conocidos (prefiltro ambiental) | Consulta NASA POWER y Earth Engine en vivo | `model/VALIDATION.md`, sección A. Se regenera con `python -m model.validate` |
| 9 de 11 cráteres mexicanos conocidos | Se midió en Colab con el detector completo | `informe/validacion_detector.pdf` |
| La Luna emborronada: 0.058 → 0.043 | Experimento en Colab con el relieve de LOLA | `model/NIVEL2.md`, sección B · `analisis/24_experimento_instrumento_marte.py` |
| Apolo 14 en El Pinacate; Apolo 15 en Colton | Son citas, no cálculos | NASA SP-2015-626 (Phinney 2015); USGS OFR 2005-1190 |
| Al Hutaymah y Jabal Bayda no aparecen como análogos | Es una búsqueda en la literatura | `model/NIVEL2.md`, sección A |

La celda siguiente solo muestra las dos líneas de `model/VALIDATION.md` con el resultado del prefiltro.

In [11]:
for linea in VALIDACION.splitlines():
    if linea.startswith("### Controles negativos") or linea.startswith("### Análogos conocidos"):
        print(linea.lstrip("# "))

Controles negativos: 3 de 3 pasan
Análogos conocidos: 9 de 21 pasan


In [12]:
# Comprobación: lo que sale aquí es lo mismo que está escrito en model/VALIDATION.md
todos, grandes = exactitud.iloc[0], exactitud.iloc[2]
esperado = [f"{int(todos.Luna):,} cráteres lunares y {int(todos.Marte)} marcianos",
            f"| D ≥ 2 km | 3 variables | {grandes.Luna} | {grandes.Marte} | {grandes['exactitud balanceada']} |",
            f"puntaje de forma {validate.auc(nat.forma_mejor, neg.forma_mejor):.2f} (p = {validate.p_permutacion(nat.forma_mejor, neg.forma_mejor):.2f})",
            f"encuentra el {c['detectado']:.0%} y el {c['pasa']:.0%} pasa el filtro",
            f"la IA etiqueta {int(vc.reindex(validate.NATURALES_IA).fillna(0).sum())} como cráteres naturales"]
for linea in esperado:
    print("✓" if linea in VALIDACION else "✗ NO COINCIDE", linea)
assert all(linea in VALIDACION for linea in esperado), "Este cuaderno y model/VALIDATION.md ya no dicen lo mismo"

✓ 4,288 cráteres lunares y 573 marcianos
✓ | D ≥ 2 km | 3 variables | 788 | 125 | 77.6% ± 0.7% |
✓ puntaje de forma 0.42 (p = 0.43)
✓ encuentra el 40% y el 32% pasa el filtro
✓ la IA etiqueta 27 como cráteres naturales
